In [1]:
import re, json, os
from typing import List, Dict

from dotenv import load_dotenv
load_dotenv()

#os.environ['OPENAI_API_KEY'] = ""


input_file = "../dataset/naturalized/original_train.json"
output_file = "../dataset/english_naturalized/train.json"

with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    request_timeout=60,
)


In [2]:
def normalize_speaker_labels(raw_dialogue: str) -> str:
    """
    줄 시작의 '화자1:' / '화자2:' 만 'Speaker1:' / 'Speaker2:' 로 치환.
    본문 내 '화자1', '화자2' 텍스트는 변경하지 않음.
    """
    s = re.sub(r'(?m)^\s*화자1\s*:', 'Speaker1:', raw_dialogue)
    s = re.sub(r'(?m)^\s*화자2\s*:', 'Speaker2:', s)
    return s

def parse_dialogue_to_turns(dialogue_str: str) -> List[Dict[str, str]]:
    """
    "Speaker1: ..." / "Speaker2: ..." 라인들을 [{'speaker': 'Speaker1', 'text': '...'}, ...] 로 변환
    """
    turns = []
    for line in dialogue_str.splitlines():
        line = line.strip()
        if not line:
            continue
        m = re.match(r'^(Speaker1|Speaker2)\s*:\s*(.*)$', line)
        if m:
            speaker, text = m.group(1), m.group(2).strip()
            turns.append({"speaker": speaker, "text": text})
    return turns


In [3]:
SYSTEM_PROMPT = (
    "You are a precise but COLLOQUIAL translator. Read the short Korean dialogue context, then translate "
    "ONLY the final utterance into natural American English that sounds like real texting/chat.\n"
    "\n"
    "Style rules:\n"
    "- Use casual, idiomatic English (contractions; light slang when it fits: yeah/yup, kinda/sorta, yikes, dang, for real).\n"
    "- Mirror the speaker’s vibe (dry vs. warm, emphatic vs. neutral). If age/gender are unclear, default to a neutral young-adult casual tone.\n"
    "- **Omission policy:** If subject/time/place or other info is obvious from the immediate context, DROP it. Prefer short, punchy lines (fragments OK). Keep new/contrastive info, negation, tense/modality.\n"
    "- Map common KR chat/emotes to TEXT-ONLY expressions (no emojis or emoticons):\n"
    "  • ㅋㅋ/ㅋㅋㅋ/ㅎㅎ → lol / haha / lmao (stronger)\n"
    "  • ㅠㅠ/ㅜㅜ → ugh / I’m bummed / this sucks (pick what fits)\n"
    "  • ㅇㅇ → yeah/yup   • 헐 → whoa/yikes/dang   • ㄴㄴ → nah   • ㄹㅇ → for real / fr\n"
    "- Keep proper nouns and numbers when they are NEW in the final line. Do NOT add info. No emojis or emoticons.\n"
    "- Output ONLY the English translation of the FINAL utterance. No explanations.\n"
    "\n"
    "Few-shot 1 (dataset tone; omit subject/time):\n"
    "Context:\n"
    "Speaker2: 진짜 신의 한수\n"
    "Speaker1: 이사하자마자 비 많이 와서 베란다 물 많이 새는 거 알았잖아\n"
    "Speaker2: 글치 계속 해떴으면 몰랐겠지\n"
    "Final utterance (Korean):\n"
    "Speaker1: 오늘 비가 엄청 많이 내리네\n"
    "Output:\n"
    "Coming down hard today.\n"
    "\n"
    "Few-shot 2 (decision vibe; omit ‘I’ since it’s obvious):\n"
    "Context:\n"
    "Speaker1: 요 아래 씽크홀 공사하던데 괜찮을라나\n"
    "Speaker2: 그러게 저번에도 비 많이 와서 땅꺼진 건데 큰일이네\n"
    "Speaker1: 하수도 공사도 같이 하더만 물 안빠져서\n"
    "Final utterance (Korean):\n"
    "Speaker2: 비 많이 올 때는 그쪽으로 다니지 말아야겠다\n"
    "Output:\n"
    "Not taking that way when it pours.\n"
    "\n"
    "Few-shot 3 (laughter ㅋㅋ; keep it tight, omit ‘you’):\n"
    "Context:\n"
    "Speaker1: 저번에 지나가다 보니 좀 무섭더라\n"
    "Speaker2: 나도 봤는데 씽크홀 크기가 엄청나더라\n"
    "Final utterance (Korean):\n"
    "Speaker1: ㅇㅇ 조심해 ㅋㅋ\n"
    "Output:\n"
    "Yeah, be careful lol.\n"
    "\n"
    "Few-shot 4 (ㅠㅠ → text-only; omit repeated ‘today’ if context already has it):\n"
    "Context:\n"
    "Speaker2: 비 많이 올 때는 그쪽으로 다니지 말아야겠다\n"
    "Speaker1: 부실공사지 뭐\n"
    "Final utterance (Korean):\n"
    "Speaker2: 오늘도 비라니 ㅠㅠ\n"
    "Output:\n"
    "Ugh, raining again.\n"
    "\n"
    "Few-shot 5 (short interjection; keep fragment):\n"
    "Context:\n"
    "Speaker1: 하수도 공사도 같이 하더만 물 안빠져서\n"
    "Speaker2: 새로 지은 곳인데도 그러네\n"
    "Final utterance (Korean):\n"
    "Speaker1: 부실공사지 뭐\n"
    "Output:\n"
    "Shoddy work, basically.\n"
    "\n"
    "Now, follow the same style for the next input."
)

# === 단일 발화 번역용 메시지 빌더 (신규) ===
def build_messages_for_single_turn(context_turns: List[Dict[str, str]],
                                   final_turn: Dict[str, str],
                                   k_context: int = 4):
    """
    - context_turns: 직전 컨텍스트 발화들(길이 <= k_context)
    - final_turn: 지금 번역할 타겟 발화 {'speaker':..., 'text':...}
    반환: OpenAI(Chat) 포맷 messages
    """
    # 컨텍스트 문자열
    ctx_lines = [f"{t['speaker']}: {t['text']}" for t in context_turns]
    context_block = "\n".join(ctx_lines) if ctx_lines else "(no previous context)"

    user_prompt = (
        f"Context (up to last {k_context} utterances):\n"
        f"{context_block}\n\n"
        f"Final utterance (Korean):\n"
        f"{final_turn['speaker']}: {final_turn['text']}\n\n"
        f"Task: Translate the FINAL utterance to English. "
        f"Return ONLY the translation."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ]

# === 전체 대화를 '한 발화씩' 순차 번역하고, 영어 대화 문자열로 반환 (신규) ===
def translate_dialogue_all_utterances(dialogue_str: str, k_context: int = 4,
                                      max_tries: int = 3, base_wait: float = 1.5) -> str:
    """
    대화를 순회하며 매 차례:
      - 직전 최대 k_context개의 발화를 컨텍스트로 제공
      - 현재(마지막) 발화만 번역
    최종적으로 "SpeakerX: <영문>" 줄들의 조합으로 하나의 문자열을 반환.
    """
    norm = normalize_speaker_labels(dialogue_str)
    turns = parse_dialogue_to_turns(norm)
    if not turns:
        return ""

    translated_lines = []

    for i, turn in enumerate(turns):
        # 직전 k개 컨텍스트
        ctx = turns[max(0, i - k_context): i]
        messages = build_messages_for_single_turn(ctx, turn, k_context=k_context)

        # 간단 재시도
        last_err = None
        for t in range(max_tries):
            try:
                resp = llm.invoke(messages)
                eng = (resp.content or "").strip()
                translated_lines.append(f"{turn['speaker']}: {eng}")
                break
            except Exception as e:
                last_err = e
                time.sleep(base_wait * (t + 1))
        else:
            # 모든 재시도 실패 시 빈 줄/오류 마킹
            translated_lines.append(f"{turn['speaker']}: ")
    
    return "\n".join(translated_lines)

In [4]:
total_utterances = 0
for rec in data:
    dlg = rec.get("dialogue", "") or ""
    if dlg.strip():
        norm = normalize_speaker_labels(dlg)
        turns = parse_dialogue_to_turns(norm)
        total_utterances += len(turns)

print(f"Total utterances to translate: {total_utterances}")

Total utterances to translate: 2109


## dialogue 항목 번역

In [5]:
import os, json, time
from tqdm.auto import tqdm

K_CONTEXT = 4
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# 데이터 로딩
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)
assert isinstance(data, list), "입력 JSON은 리스트여야 합니다."

processed = []
n_err = 0

for rec in tqdm(data, desc="Translating all utterances (k=4)", ncols=90):
    dlg = rec.get("dialogue", "") or ""
    try:
        if dlg.strip():
            # 전체 발화를 순차 번역하고, 번역 결과를 dialogue에 덮어쓰기
            translated_dialogue = translate_dialogue_all_utterances(dlg, k_context=K_CONTEXT)
            rec["dialogue"] = translated_dialogue
        else:
            rec["dialogue"] = ""
    except Exception as e:
        n_err += 1
        # 실패 시 원문 유지하거나 빈 문자열로 처리(여기선 원문 유지 선택)
        rec["dialogue"] = dlg
        rec["translation_error"] = str(e)
    processed.append(rec)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(processed, f, ensure_ascii=False, indent=2)

print(f"Done. total={len(processed)}, errors={n_err}, saved -> {output_file}")

/opt/anaconda3/envs/multi-agent/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Translating all utterances (k=4): 100%|█████████████████| 504/504 [32:49<00:00,  3.91s/it]

Done. total=504, errors=0, saved -> ../dataset/english_naturalized/train.json
